# Theta-only Baseline vs Theta + NV Embedding

这个 notebook 对比两个 run：

1. `dcedge20_2724`: `theta-only baseline`
2. `theta_nv_embed_2740`: `theta + nv embedding`

目标：

- 先核对两组实验是否可以作为合理对照
- 再用 `preds.npz` 中的 `logE_true`、`logE_pred`、`mc_weight` 重新计算并叠加两条 weighted 曲线
- 输出三张对比图：`resolution`、`bias`、`RMS`


## 1. 环境准备

In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/home/server/projects/energy_reconstruction')
NOTEBOOK_DIR = PROJECT_ROOT / 'notebook'
OUTPUT_DIR = NOTEBOOK_DIR / 'generated' / 'theta_only_vs_nv_embed_2724_2740'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR   =', OUTPUT_DIR)

## 2. 定义两个实验目录并检查结果文件

In [ ]:
RUNS = {
    'theta_only': {
        'run_name': 'dcedge20_2724',
        'display': 'theta-only baseline',
    },
    'theta_plus_nv': {
        'run_name': 'theta_nv_embed_2740',
        'display': 'theta + nv embedding',
    },
}

RESULTS = {}
for key, item in RUNS.items():
    run_dir = PROJECT_ROOT / 'runs' / item['run_name']
    fig_dir = run_dir / 'fig'
    config_path = run_dir / 'config.json'
    metrics_path = fig_dir / 'metrics.json'
    preds_path = fig_dir / 'preds.npz'
    train_stats_path = run_dir / 'dataset_train_stats.json'
    test_stats_path = run_dir / 'dataset_test_stats.json'

    assert run_dir.exists(), f'Missing run dir: {run_dir}'
    assert config_path.exists(), f'Missing config: {config_path}'
    assert metrics_path.exists(), f'Missing metrics: {metrics_path}'
    assert preds_path.exists(), f'Missing preds: {preds_path}'
    assert train_stats_path.exists(), f'Missing train stats: {train_stats_path}'
    assert test_stats_path.exists(), f'Missing test stats: {test_stats_path}'

    RESULTS[key] = {
        'display': item['display'],
        'run_dir': run_dir,
        'config_path': config_path,
        'metrics_path': metrics_path,
        'preds_path': preds_path,
        'train_stats_path': train_stats_path,
        'test_stats_path': test_stats_path,
    }

for key, item in RESULTS.items():
    print(f"{key:14s} -> {item['run_dir']}")
    print(f"  config      : {item['config_path']}")
    print(f"  metrics     : {item['metrics_path']}")
    print(f"  preds       : {item['preds_path']}")
    print(f"  train stats : {item['train_stats_path']}")
    print(f"  test stats  : {item['test_stats_path']}")

## 3. 读取结果文件

In [ ]:
loaded = {}
for key, item in RESULTS.items():
    with open(item['config_path'], 'r') as f:
        config = json.load(f)
    with open(item['metrics_path'], 'r') as f:
        metrics = json.load(f)
    with open(item['train_stats_path'], 'r') as f:
        train_stats = json.load(f)
    with open(item['test_stats_path'], 'r') as f:
        test_stats = json.load(f)
    preds = np.load(item['preds_path'])

    loaded[key] = {
        'display': item['display'],
        'config': config,
        'metrics': metrics,
        'train_stats': train_stats,
        'test_stats': test_stats,
        'logE_true': preds['logE_true'].astype(np.float64),
        'logE_pred': preds['logE_pred'].astype(np.float64),
        'mc_weight': preds['mc_weight'].astype(np.float64),
    }

for key, item in loaded.items():
    print(item['display'])
    print('  n eval      :', item['metrics']['n'])
    print('  w_log_sigma :', item['metrics']['w_log_sigma'])
    print('  w_log_rmse  :', item['metrics']['w_log_rmse'])
    print('  w_log_bias  :', item['metrics']['w_log_bias'])

## 4. 检查实验一致性

In [ ]:
compare_keys = [
    'root_path', 'n_files', 'seed', 'test_size', 'val_size', 'max_points', 'batch_size', 'num_workers', 'pin_memory', 'io_workers',
    'Emin', 'Emax', 'pinc_max', 'dcedge_min', 'dangle_max_deg', 'theta_max_deg', 'use_core_box', 'core_box', 'vqsamp_ratio_min',
    'norm_mode', 'sample_mode', 'epochs', 'lr', 'patience', 'min_delta', 'grad_clip', 'bins_hist', 'min_count', 'max_weight',
    'loss_mode', 'huber_delta', 'rel_delta', 'rel_squared', 'eval_space', 'save_arrays', 'theta_embed_dim', 'theta_embed_dropout'
]

diff_rows = []
for key in compare_keys:
    a = loaded['theta_only']['config'].get(key)
    b = loaded['theta_plus_nv']['config'].get(key)
    if a != b:
        diff_rows.append((key, a, b))

print('Comparable config diffs:')
if diff_rows:
    for key, a, b in diff_rows:
        print(f'  {key}: theta_only={a!r} | theta_plus_nv={b!r}')
else:
    print('  None')

print('
Other config/schema diffs:')
all_keys = sorted(set(loaded['theta_only']['config']) | set(loaded['theta_plus_nv']['config']))
for key in all_keys:
    a = loaded['theta_only']['config'].get(key, '<MISSING>')
    b = loaded['theta_plus_nv']['config'].get(key, '<MISSING>')
    if a != b and key not in compare_keys:
        print(f'  {key}: theta_only={a!r} | theta_plus_nv={b!r}')

print('
Dataset stats notes:')
print('  theta_only train point_center =', loaded['theta_only']['train_stats'].get('point_center', '<not recorded>'))
print('  theta_plus_nv train point_center =', loaded['theta_plus_nv']['train_stats'].get('point_center', '<not recorded>'))
print('  theta_only test n_fail =', loaded['theta_only']['test_stats']['files']['n_fail'])
print('  theta_plus_nv test n_fail =', loaded['theta_plus_nv']['test_stats']['files']['n_fail'])

print('
Conclusion: if the only comparable config diffs are nv-related, this is a reasonable control comparison.')

## 5. 用共享 true-energy bins 重算三条 weighted 曲线

In [ ]:
def weighted_mean(x, w):
    w = np.asarray(w, dtype=np.float64)
    x = np.asarray(x, dtype=np.float64)
    s = np.sum(w)
    return np.nan if s <= 0 else np.sum(w * x) / s


def weighted_var(x, w):
    m = weighted_mean(x, w)
    if not np.isfinite(m):
        return np.nan
    w = np.asarray(w, dtype=np.float64)
    x = np.asarray(x, dtype=np.float64)
    s = np.sum(w)
    return np.nan if s <= 0 else np.sum(w * (x - m) ** 2) / s


def weighted_std(x, w):
    v = weighted_var(x, w)
    return np.sqrt(v) if np.isfinite(v) and v >= 0 else np.nan


def weighted_rms(x, w):
    w = np.asarray(w, dtype=np.float64)
    x = np.asarray(x, dtype=np.float64)
    s = np.sum(w)
    return np.nan if s <= 0 else np.sqrt(np.sum(w * x**2) / s)


def compute_weighted_curves(log_true, log_pred, weights, bin_edges):
    mask = (
        np.isfinite(log_true)
        & np.isfinite(log_pred)
        & np.isfinite(weights)
        & (weights > 0)
    )
    log_true = log_true[mask]
    log_pred = log_pred[mask]
    weights = weights[mask]

    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bias = []
    resolution = []
    log_rms = []

    for i in range(len(bin_edges) - 1):
        m = (log_true >= bin_edges[i]) & (log_true < bin_edges[i + 1])
        if m.sum() > 10:
            residual = log_pred[m] - log_true[m]
            w = weights[m]
            bias.append(float(weighted_mean(residual, w)))
            resolution.append(float(weighted_std(log_pred[m], w)))
            log_rms.append(float(weighted_rms(residual, w)))
        else:
            bias.append(np.nan)
            resolution.append(np.nan)
            log_rms.append(np.nan)

    return {
        'bin_centers': np.asarray(centers, dtype=np.float64),
        'bias': np.asarray(bias, dtype=np.float64),
        'resolution': np.asarray(resolution, dtype=np.float64),
        'log_rms': np.asarray(log_rms, dtype=np.float64),
    }

combined_true = np.concatenate([
    loaded['theta_only']['logE_true'][np.isfinite(loaded['theta_only']['logE_true'])],
    loaded['theta_plus_nv']['logE_true'][np.isfinite(loaded['theta_plus_nv']['logE_true'])],
])
bin_edges = np.linspace(combined_true.min(), combined_true.max(), 21)

curves = {}
for key, item in loaded.items():
    curves[key] = compute_weighted_curves(
        item['logE_true'],
        item['logE_pred'],
        item['mc_weight'],
        bin_edges,
    )
    print(item['display'], 'valid bins =', np.isfinite(curves[key]['bias']).sum())

## 6. 定义统一绘图函数

In [ ]:
plot_order = ['theta_only', 'theta_plus_nv']
colors = {
    'theta_only': '#1f77b4',
    'theta_plus_nv': '#d62728',
}
markers = {
    'theta_only': 'o',
    'theta_plus_nv': 's',
}

def plot_compare(metric_key, ylabel, title, filename, draw_zero=False):
    fig, ax = plt.subplots(figsize=(8, 5))
    for key in plot_order:
        item = loaded[key]
        curve = curves[key]
        ax.plot(
            curve['bin_centers'],
            curve[metric_key],
            marker=markers[key],
            color=colors[key],
            linewidth=2,
            markersize=5,
            label=item['display'],
        )

    if draw_zero:
        ax.axhline(0.0, color='gray', linestyle='--', linewidth=1)

    ax.set_xlabel(r'True Energy $\log_{10}(E/\mathrm{GeV})$')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=True)
    ax.grid(alpha=0.3)
    fig.tight_layout()

    out_path = OUTPUT_DIR / filename
    fig.savefig(out_path, dpi=200)
    plt.show()
    print('saved:', out_path)
    return out_path

## 7. 画 `resolution` 对比图

In [ ]:
resolution_path = plot_compare(
    metric_key='resolution',
    ylabel='Weighted resolution',
    title='Theta-only baseline vs theta + nv embedding: weighted resolution',
    filename='resolution_theta_only_vs_nv_embed_2724_2740.png',
)
resolution_path

## 8. 画 `bias` 对比图

In [ ]:
bias_path = plot_compare(
    metric_key='bias',
    ylabel='Weighted bias',
    title='Theta-only baseline vs theta + nv embedding: weighted bias',
    filename='bias_theta_only_vs_nv_embed_2724_2740.png',
    draw_zero=True,
)
bias_path

## 9. 画 `RMS` 对比图

In [ ]:
rms_path = plot_compare(
    metric_key='log_rms',
    ylabel='Weighted log RMS error',
    title='Theta-only baseline vs theta + nv embedding: weighted log RMS',
    filename='logRMS_theta_only_vs_nv_embed_2724_2740.png',
)
rms_path

## 10. 小结

In [ ]:
summary = {
    'theta_only': str(RESULTS['theta_only']['run_dir']),
    'theta_plus_nv': str(RESULTS['theta_plus_nv']['run_dir']),
    'judgement': 'dcedge20_2724 is theta-only baseline; theta_nv_embed_2740 is theta + nv embedding',
    'outputs': {
        'resolution': str(resolution_path),
        'bias': str(bias_path),
        'rms': str(rms_path),
    },
}
summary